In [41]:
import pandas as pd

In [42]:
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'

In [43]:
columns = ['lpep_pickup_datetime',
'lpep_dropoff_datetime',
'PULocationID',
'DOLocationID',
'passenger_count',
'trip_distance',
'tip_amount',
'total_amount']
df = pd.read_parquet(url, columns=columns)

In [44]:
from models import Ride, ride_from_row, ride_serializer

In [45]:
ride = ride_from_row(df.iloc[1])
ride

Ride(lpep_pickup_datetime=1759277643000, lpep_dropoff_datetime=1759278254000, PULocationID=66, DOLocationID=25, passenger_count=1, trip_distance=1.61, tip_amount=2.78, total_amount=16.68)

In [46]:
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [47]:
len(df)

49416

In [50]:
from time import time

t0 = time()

topic_name = 'green-trips'

producer.send(topic_name, value=ride)
producer.flush()


t1 = time()
print(f'took {(t1 - t0):.2f} seconds')

took 0.01 seconds


In [26]:
import time
topic_name = 'green-trips'

t0 = time.time()
i=0

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    #print(f"Sent: {ride}")
    #time.sleep(0.01)
    i+=1
    if i%10000==0:
        print(f"зашло строк:{i}")

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

зашло строк:10000
зашло строк:20000
зашло строк:30000
зашло строк:40000
took 10.29 seconds
